# motif-discover vs STREME: Live Benchmark

**motif-discover runs live** — you'll see it discover motifs in seconds. STREME comparison data is from the full 132-TF benchmark (included in `benchmark_data/`).

## 1. Setup

In [ ]:
import os, subprocess, time

if not os.path.isdir('/content/motif-discover/.git'):
    subprocess.run('rm -rf /content/motif-discover && git clone -q https://github.com/Travis42/motif-discover.git /content/motif-discover', shell=True)
os.chdir('/content/motif-discover')
subprocess.run(['chmod', '+x', 'motif-discover'])

n_tfs = len([f for f in os.listdir('example') if f.endswith('.fa')])
print(f'motif-discover: {os.path.getsize("motif-discover")//1024} KB (static binary, zero deps)')
print(f'Example data:   {n_tfs} TFs')

## 2. Run motif-discover Live

In [ ]:
t0 = time.time()
r = subprocess.run(
    ['./motif-discover', '--data', 'example/', '--ours-only', '--no-meme'],
    capture_output=True, text=True
)
elapsed = time.time() - t0
print(f'Wall time: {elapsed:.1f}s for {n_tfs} TFs')
print(r.stdout)

## 3. motif-discover Results

In [ ]:
import pandas as pd

rows = []
for line in r.stdout.strip().split('\n'):
    parts = line.split('\t')
    if len(parts) >= 5 and parts[0] != 'TF' and parts[4] == 'ours':
        rows.append({
            'TF': parts[0], 'Width': int(parts[1]),
            'AUROC': float(parts[2]), 'Time_s': float(parts[3]),
        })

live = pd.DataFrame(rows)
print(f'Average AUROC: {live["AUROC"].mean():.4f}')
print(f'Average time:  {live["Time_s"].mean():.2f}s/TF')
live[['TF', 'Width', 'AUROC', 'Time_s']]

## 4. Full Benchmark: motif-discover vs STREME (132 TFs)

Pre-computed results from the full 132-TF ENCODE K562 benchmark. Same algorithm, same parameters — just more TFs and STREME included.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import wilcoxon

shuf = pd.read_csv('benchmark_data/shuffled_negatives_132tf.csv')
common = shuf.dropna(subset=['motif_discover_AUROC', 'STREME_AUROC'])

MD_COLOR = '#2166AC'
ST_COLOR = '#D6604D'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Scatter
ax = axes[0]
o = common['motif_discover_AUROC'].values
s = common['STREME_AUROC'].values
colors = np.where(o > s, MD_COLOR, ST_COLOR)
ax.scatter(s, o, alpha=0.5, s=25, c=colors)
ax.plot([0.4, 1.0], [0.4, 1.0], 'k--', alpha=0.3)
ax.set_xlabel('STREME AUROC')
ax.set_ylabel('motif-discover AUROC')
ax.set_title(f'Shuffled Negatives (n={len(common)}, 132 TFs)')
for _, row in common.iterrows():
    d = row['motif_discover_AUROC'] - row['STREME_AUROC']
    if abs(d) > 0.2:
        ax.annotate(row['TF'], (row['STREME_AUROC'], row['motif_discover_AUROC']), fontsize=7)

# Panel B: Speed
ax = axes[1]
md_t = common['motif_discover_Time_s'].mean()
st_t = common['STREME_Time_s'].mean()
bars = ax.bar(['motif-discover', 'STREME'], [md_t, st_t],
              color=[MD_COLOR, ST_COLOR], edgecolor='black', width=0.5)
for bar, t in zip(bars, [md_t, st_t]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{t:.2f}s', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Time per TF (seconds)')
ax.set_title(f'Speed ({st_t/md_t:.0f}x faster)')

# Panel C: Genomic negatives
gen = pd.read_csv('benchmark_data/genomic_negatives_132tf.csv')
gen_common = gen.dropna(subset=['motif_discover_AUROC', 'STREME_AUROC'])
ax = axes[2]
tools = ['motif-discover', 'STREME']
aurocs = [gen_common['motif_discover_AUROC'].mean(), gen_common['STREME_AUROC'].mean()]
bars = ax.bar(tools, aurocs, color=[MD_COLOR, ST_COLOR], edgecolor='black', width=0.5)
ax.set_ylabel('Mean AUROC')
ax.set_title(f'Genomic Negatives (n={len(gen_common)}, Markov-1)')
ax.set_ylim(0.82, 0.92)
for bar, a in zip(bars, aurocs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{a:.4f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Stats
diffs = common['motif_discover_AUROC'].values - common['STREME_AUROC'].values
_, p = wilcoxon(diffs)
wins = (diffs > 0.001).sum()
losses = (diffs < -0.001).sum()
print(f'\nmotif-discover: {common["motif_discover_AUROC"].mean():.4f} AUROC ({md_t:.2f}s/TF)')
print(f'STREME:         {common["STREME_AUROC"].mean():.4f} AUROC ({st_t:.2f}s/TF)')
print(f'Speedup:        {st_t/md_t:.1f}x')
print(f'Wilcoxon p:     {p:.2e}  ({wins}W/{losses}L)')

---

Cells 2–3 run motif-discover live on this machine. Cell 4 uses pre-computed 132-TF benchmark data (included in the repo). Full paper: [github.com/Travis42/motif-discover](https://github.com/Travis42/motif-discover)